In [ ]:
# MarketMind Intelligence Platform V1
# Author: Sharique Mohammad
# Date: April 23, 2026

# MarketMind V1 - Data Quality Validation

**Notebook:** 05_data_quality_validation.ipynb  
**Purpose:** Validate data completeness, consistency, and integrity across all Gold tables

---

## Overview

This notebook performs comprehensive data quality checks:
- Completeness validation (missing values, coverage gaps)
- Consistency checks (duplicates, invalid values)
- Referential integrity
- Statistical anomaly detection
- Data freshness assessment

**Tables Validated:**
- gold.ohlcv_bars
- gold.macro_indicators
- gold.sec_filings
- gold.corporate_actions

In [ ]:
# Import libraries
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Set style
plt.style.use('seaborn-v0_8-whitegrid')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('Libraries imported successfully!')

## 1. Database Connection

In [ ]:
# PostgreSQL connection
DB_CONFIG = {
    'host': '172.31.32.1',
    'port': 5432,
    'database': 'marketmind_v1',
    'user': 'postgres',
    'password': '0940'
}

print('Database configuration loaded')

## 2. OHLCV Data Quality Checks

In [ ]:
conn = psycopg2.connect(**DB_CONFIG)

# Load OHLCV data
query = "SELECT * FROM gold.ohlcv_bars"
df_ohlcv = pd.read_sql(query, conn)
df_ohlcv['date'] = pd.to_datetime(df_ohlcv['date'])

conn.close()

print('OHLCV Data Quality Report')
print('=' * 80)

# 1. Record count
print(f'Total Records: {len(df_ohlcv):,}')
print(f'Unique Tickers: {df_ohlcv["ticker"].nunique()}')
print(f'Date Range: {df_ohlcv["date"].min()} to {df_ohlcv["date"].max()}')
print()

# 2. Null value analysis
print('NULL Value Analysis:')
null_counts = df_ohlcv.isnull().sum()
null_pct = (null_counts / len(df_ohlcv)) * 100
null_df = pd.DataFrame({'Null Count': null_counts, 'Null %': null_pct})
null_df = null_df[null_df['Null Count'] > 0]
if len(null_df) > 0:
    print(null_df)
else:
    print('  No NULL values in required columns')
print()

# 3. Duplicate check
duplicates = df_ohlcv.duplicated(subset=['ticker', 'timestamp']).sum()
print(f'Duplicate Records (ticker + timestamp): {duplicates}')
if duplicates > 0:
    print('  STATUS: FAIL - Duplicates found!')
else:
    print('  STATUS: PASS')
print()

# 4. Price validation (OHLC consistency)
invalid_ohlc = df_ohlcv[
    (df_ohlcv['high'] < df_ohlcv['low']) | 
    (df_ohlcv['high'] < df_ohlcv['open']) |
    (df_ohlcv['high'] < df_ohlcv['close']) |
    (df_ohlcv['low'] > df_ohlcv['open']) |
    (df_ohlcv['low'] > df_ohlcv['close'])
]
print(f'Invalid OHLC Records (high < low, etc.): {len(invalid_ohlc)}')
if len(invalid_ohlc) > 0:
    print('  STATUS: FAIL - Invalid OHLC data!')
    print(invalid_ohlc[['ticker', 'date', 'open', 'high', 'low', 'close']].head())
else:
    print('  STATUS: PASS')
print()

# 5. Date coverage gaps
print('Date Coverage per Ticker:')
coverage = df_ohlcv.groupby('ticker')['date'].agg(['min', 'max', 'count'])
coverage['expected_days'] = (coverage['max'] - coverage['min']).dt.days + 1
coverage['coverage_pct'] = (coverage['count'] / coverage['expected_days']) * 100
print(coverage[['count', 'expected_days', 'coverage_pct']].to_string())
print()

print('=' * 80)

## 3. OHLCV Data Quality Visualizations

In [ ]:
# Visualize NULL percentages
null_summary = (df_ohlcv.isnull().sum() / len(df_ohlcv)) * 100
null_summary = null_summary[null_summary > 0].sort_values(ascending=False)

if len(null_summary) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(range(len(null_summary)), null_summary.values, color='coral', edgecolor='black')
    ax.set_yticks(range(len(null_summary)))
    ax.set_yticklabels(null_summary.index)
    ax.set_xlabel('NULL Percentage (%)', fontsize=12, fontweight='bold')
    ax.set_title('OHLCV Data - NULL Value Analysis', fontsize=14, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    
    for bar, val in zip(bars, null_summary.values):
        ax.text(val + 0.5, bar.get_y() + bar.get_height()/2.,
                f'{val:.2f}%',
                ha='left', va='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
else:
    print('No NULL values to visualize - all required columns complete!')

# Visualize data coverage
fig, ax = plt.subplots(figsize=(12, 6))
coverage_sorted = coverage.sort_values('coverage_pct')
colors = ['green' if x >= 95 else 'orange' if x >= 90 else 'red' for x in coverage_sorted['coverage_pct']]
bars = ax.barh(range(len(coverage_sorted)), coverage_sorted['coverage_pct'], color=colors, edgecolor='black')

ax.set_yticks(range(len(coverage_sorted)))
ax.set_yticklabels(coverage_sorted.index)
ax.set_xlabel('Coverage (%)', fontsize=12, fontweight='bold')
ax.set_title('OHLCV Data Coverage by Ticker', fontsize=14, fontweight='bold')
ax.axvline(95, color='darkgreen', linestyle='--', linewidth=2, label='95% Threshold')
ax.legend()
ax.grid(axis='x', alpha=0.3)

for bar, val in zip(bars, coverage_sorted['coverage_pct']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2.,
            f'{val:.1f}%',
            ha='left', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Macro Indicators Data Quality Checks

In [ ]:
conn = psycopg2.connect(**DB_CONFIG)

# Load macro data
query = "SELECT * FROM gold.macro_indicators"
df_macro = pd.read_sql(query, conn)
df_macro['date'] = pd.to_datetime(df_macro['date'])

conn.close()

print('Macro Indicators Data Quality Report')
print('=' * 80)

# 1. Record count
print(f'Total Records: {len(df_macro):,}')
print(f'Unique Indicators: {df_macro["indicator_name"].nunique()}')
print(f'Date Range: {df_macro["date"].min()} to {df_macro["date"].max()}')
print()

# 2. NULL value analysis by indicator
print('NULL Value Analysis by Indicator:')
for indicator in df_macro['indicator_name'].unique():
    ind_data = df_macro[df_macro['indicator_name'] == indicator]
    null_count = ind_data['value'].isnull().sum()
    null_pct = (null_count / len(ind_data)) * 100
    print(f'  {indicator}: {null_count} NULLs ({null_pct:.1f}%)')
print()

# 3. Duplicate check
duplicates = df_macro.duplicated(subset=['indicator_name', 'date']).sum()
print(f'Duplicate Records (indicator + date): {duplicates}')
if duplicates > 0:
    print('  STATUS: FAIL - Duplicates found!')
else:
    print('  STATUS: PASS')
print()

# 4. Data freshness
latest_dates = df_macro.groupby('indicator_name')['date'].max()
print('Latest Data Points:')
for indicator, date in latest_dates.items():
    days_old = (pd.Timestamp.now() - date).days
    print(f'  {indicator}: {date.strftime("%Y-%m-%d")} ({days_old} days old)')
print()

print('=' * 80)

## 5. SEC Filings Data Quality Checks

In [ ]:
conn = psycopg2.connect(**DB_CONFIG)

# Load SEC filings data
query = "SELECT * FROM gold.sec_filings"
df_filings = pd.read_sql(query, conn)
df_filings['filing_date'] = pd.to_datetime(df_filings['filing_date'])
df_filings['report_date'] = pd.to_datetime(df_filings['report_date'])

conn.close()

print('SEC Filings Data Quality Report')
print('=' * 80)

# 1. Record count
print(f'Total Records: {len(df_filings):,}')
print(f'Unique Companies: {df_filings["ticker"].nunique()}')
print(f'Date Range: {df_filings["filing_date"].min()} to {df_filings["filing_date"].max()}')
print()

# 2. UNKNOWN ticker analysis
unknown_count = (df_filings['ticker'] == 'UNKNOWN').sum()
unknown_pct = (unknown_count / len(df_filings)) * 100
print(f'UNKNOWN Ticker Records: {unknown_count} ({unknown_pct:.1f}%)')
print('  Note: This is expected - EdgarTools API limitation for some companies')
print()

# 3. Duplicate check
duplicates = df_filings.duplicated(subset=['accession_number']).sum()
print(f'Duplicate Records (accession_number): {duplicates}')
if duplicates > 0:
    print('  STATUS: FAIL - Duplicates found!')
else:
    print('  STATUS: PASS')
print()

# 4. Filing lag validation
invalid_lags = df_filings[df_filings['filing_lag_days'] < 0]
print(f'Invalid Filing Lags (negative days): {len(invalid_lags)}')
if len(invalid_lags) > 0:
    print('  STATUS: FAIL - Negative lags found!')
    print(invalid_lags[['ticker', 'form_type', 'filing_date', 'report_date', 'filing_lag_days']].head())
else:
    print('  STATUS: PASS')
print()

# 5. Form type distribution
print('Form Type Distribution:')
form_dist = df_filings['form_type'].value_counts()
for form, count in form_dist.items():
    pct = (count / len(df_filings)) * 100
    print(f'  {form}: {count} ({pct:.1f}%)')
print()

print('=' * 80)

## 6. Corporate Actions Data Quality Checks

In [ ]:
conn = psycopg2.connect(**DB_CONFIG)

# Load corporate actions data
query = "SELECT * FROM gold.corporate_actions"
df_actions = pd.read_sql(query, conn)

conn.close()

print('Corporate Actions Data Quality Report')
print('=' * 80)

if len(df_actions) == 0:
    print('Total Records: 0')
    print('STATUS: No corporate actions in dataset (expected for Q1 2026)')
else:
    print(f'Total Records: {len(df_actions):,}')
    print(f'Unique Tickers: {df_actions["ticker"].nunique()}')
    print()
    
    # Duplicate check
    duplicates = df_actions.duplicated(subset=['ticker', 'action_type', 'execution_date']).sum()
    print(f'Duplicate Records: {duplicates}')
    if duplicates > 0:
        print('  STATUS: FAIL - Duplicates found!')
    else:
        print('  STATUS: PASS')
    print()
    
    # Action type distribution
    print('Action Type Distribution:')
    action_dist = df_actions['action_type'].value_counts()
    for action, count in action_dist.items():
        print(f'  {action}: {count}')

print('=' * 80)

## 7. Overall Data Quality Summary

In [ ]:
# Create summary scorecard
summary = {
    'Table': ['ohlcv_bars', 'macro_indicators', 'sec_filings', 'corporate_actions'],
    'Record Count': [
        len(df_ohlcv),
        len(df_macro),
        len(df_filings),
        len(df_actions)
    ],
    'Duplicates': [
        df_ohlcv.duplicated(subset=['ticker', 'timestamp']).sum(),
        df_macro.duplicated(subset=['indicator_name', 'date']).sum(),
        df_filings.duplicated(subset=['accession_number']).sum(),
        df_actions.duplicated().sum() if len(df_actions) > 0 else 0
    ],
    'Status': [
        'PASS' if df_ohlcv.duplicated(subset=['ticker', 'timestamp']).sum() == 0 else 'FAIL',
        'PASS' if df_macro.duplicated(subset=['indicator_name', 'date']).sum() == 0 else 'FAIL',
        'PASS' if df_filings.duplicated(subset=['accession_number']).sum() == 0 else 'FAIL',
        'PASS' if len(df_actions) == 0 or df_actions.duplicated().sum() == 0 else 'FAIL'
    ]
}

summary_df = pd.DataFrame(summary)

print('\nData Quality Summary Scorecard')
print('=' * 80)
print(summary_df.to_string(index=False))
print('=' * 80)

# Visualize summary
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Record counts
ax1.bar(summary_df['Table'], summary_df['Record Count'], color='steelblue', edgecolor='black')
ax1.set_ylabel('Record Count', fontsize=12, fontweight='bold')
ax1.set_title('Record Counts by Table', fontsize=14, fontweight='bold')
ax1.set_xticklabels(summary_df['Table'], rotation=45, ha='right')
ax1.grid(axis='y', alpha=0.3)

# Quality status
colors = ['green' if x == 'PASS' else 'red' for x in summary_df['Status']]
ax2.bar(summary_df['Table'], [1 if x == 'PASS' else 0 for x in summary_df['Status']], 
        color=colors, edgecolor='black')
ax2.set_ylabel('Quality Status', fontsize=12, fontweight='bold')
ax2.set_title('Data Quality Status', fontsize=14, fontweight='bold')
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['FAIL', 'PASS'])
ax2.set_xticklabels(summary_df['Table'], rotation=45, ha='right')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()